In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load data

In [ ]:
train = pd.read_csv("/kaggle/input/datasets/ahmedamr11/titnaic/Titanic_train.csv")
test  = pd.read_csv("/kaggle/input/datasets/ahmedamr11/titnaic/Titanic_test.csv")

In [ ]:
print("Train shape:", train.shape)
print("Test  shape:", test.shape)

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
train.info()

In [ ]:
train.describe()

# 2) Data cleaning

In [ ]:
print("Missing values – Train:")
print(train.isnull().sum())
print()
print("Missing values – Test:")
print(test.isnull().sum())

In [ ]:
train['Age']      = train['Age'].fillna(train['Age'].median())
test['Age']       = test['Age'].fillna(test['Age'].median())
test['Fare']      = test['Fare'].fillna(test['Fare'].median())
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])
test['Embarked']  = test['Embarked'].fillna(test['Embarked'].mode()[0])

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

In [ ]:

plt.figure(figsize=(6, 5))


sns.countplot(data=train, x='Survived', hue='Survived', palette='Set2', legend=False)

plt.title('Overall Survival Count')
plt.show()

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 5))


sns.countplot(data=train, x='Pclass', hue='Survived', ax=axes[0], palette='Set2')
axes[0].set_title('Survival by Pclass')


sns.countplot(data=train, x='Sex', hue='Survived', ax=axes[1], palette='Set2')
axes[1].set_title('Survival by Sex')


sns.countplot(data=train, x='Embarked', hue='Survived', ax=axes[2], palette='Set2')
axes[2].set_title('Survival by Embarked')

plt.tight_layout()
plt.show()

In [ ]:
# Age & Fare distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train['Age'].dropna(),kde=True,  ax=axes[0], color='blue')
axes[0].set_title('Age Distribution')
sns.histplot(train['Fare'], kde=True, ax=axes[1], color='red')
axes[1].set_title('Fare Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric only)
plt.figure(figsize=(10, 6))
numeric_train = train.select_dtypes(include=[np.number])
sns.heatmap(numeric_train.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Extract Title from Name
train['Title'] = train['Name'].str.extract(r',\s*([^\.]+)\.')
test['Title']  = test['Name'].str.extract(r',\s*([^\.]+)\.')
print(train['Title'].value_counts())

In [ ]:
# Visualise survival by title
plt.figure(figsize=(10, 5))
sns.countplot(data=train, x='Title', hue='Survived', palette='Set2')
plt.title('Survival Count by Title')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Survival rate by title
survival_rate = train.groupby('Title')['Survived'].mean().sort_values(ascending=False)
survival_rate.plot(kind='bar', color='steelblue', edgecolor='black', figsize=(10, 4))
plt.title('Survival Rate by Title')
plt.ylabel('Survival Rate')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Consolidate rare titles
common_titles = ['Mr', 'Miss', 'Mrs', 'Master']
train['Title'] = train['Title'].apply(lambda x: x if x in common_titles else 'Rare')
test['Title']  = test['Title'].apply(lambda x: x if x in common_titles else 'Rare')
print(train['Title'].value_counts())
print("==================")
print(test['Title'].value_counts())

In [ ]:
# Family size & IsAlone
for df in [train, test]:
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

In [ ]:
plt.figure(figsize=(10, 6))

sns.kdeplot(data=train[train['Survived'] == 0]['Age'], label='Not Survived', fill=True, color='red')
sns.kdeplot(data=train[train['Survived'] == 1]['Age'], label='Survived', fill=True, color='green')

plt.title('Age Distribution by Survival')
plt.xlabel('Age')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
# Age bins
for df in [train, test]:
    df['AgeBin'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100],
                          labels=['Child', 'Teen', 'Young', 'Adult', 'Senior'])

In [ ]:
train.head()

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

In [ ]:
# Drop columns not needed for modelling
drop_cols = ['PassengerId', 'Ticket', 'Cabin', 'Name', 'SibSp', 'Parch']
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

In [ ]:
train.head()

In [ ]:

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, col in zip(axes, ['Age', 'Fare', 'Pclass', 'FamilySize']):
    sns.boxplot(data=train, y=col, ax=ax)
    ax.set_title(f'{col} Boxplot')

plt.tight_layout()
plt.show()

In [ ]:

train['Fare'] = train['Fare'].apply(np.log1p)
test['Fare'] = test['Fare'].apply(np.log1p)


plt.figure(figsize=(7, 5))
sns.histplot(train['Fare'], kde=True, color='green')
plt.title('Fare Distribution (After Log Transformation)')
plt.show()

In [ ]:
# Encode binary: Sex
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
test['Sex']  = test['Sex'].map({'male': 0, 'female': 1})

In [ ]:
train = pd.get_dummies(train, columns=['Embarked', 'Title'], drop_first=True)
test  = pd.get_dummies(test,  columns=['Embarked', 'Title'], drop_first=True)

In [ ]:
train.head()

In [ ]:
train.head()

In [ ]:
for df in [train, test]:
    bool_cols = df.select_dtypes(include='bool').columns
    df[bool_cols] = df[bool_cols].astype(int)

print("Train columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())

In [ ]:
train.head()

In [ ]:
# One-hot encode AgeBin
train = pd.get_dummies(train, columns=['AgeBin'], drop_first=False, prefix='AgeBin')
test = pd.get_dummies(test, columns=['AgeBin'], drop_first=False, prefix='AgeBin')

# Ensure both datasets have the same columns
train_cols = set(train.columns)
test_cols = set(test.columns)

# Add missing columns to test with 0 values
for col in train_cols - test_cols:
    test[col] = 0

# Reorder test columns to match train
test = test[train.columns]

In [ ]:
# Drop Age first
train.drop('Age', axis=1, inplace=True)
test.drop('Age', axis=1, inplace=True)

# Then split
X = train.drop('Survived', axis=1)
y = train['Survived']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

test = test[X_train.columns]

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)

In [ ]:
train.head()

In [ ]:
# Correlation heatmap (numeric only)
plt.figure(figsize=(10, 6))
numeric_train = train.select_dtypes(include=[np.number])
sns.heatmap(numeric_train.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)

lr_train_acc = accuracy_score(y_train, lr.predict(X_train))
lr_val_acc   = accuracy_score(y_val,   lr.predict(X_val))

print("=" * 50)
print("Model : Logistic Regression")
print(f"Train Accuracy : {lr_train_acc:.4f}")
print(f"Val   Accuracy : {lr_val_acc:.4f}")
print(classification_report(y_val, lr.predict(X_val)))

In [ ]:
cm = confusion_matrix(y_val, lr.predict(X_val))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Logistic Regression – Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
from sklearn.model_selection import learning_curve
train_sizes, train_scores, val_scores = learning_curve(
    lr, X_train, y_train, cv=5, scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 10))


plt.figure(figsize=(10, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), label='Train Accuracy', color='blue')
plt.plot(train_sizes, val_scores.mean(axis=1),   label='Val Accuracy',   color='orange')
plt.fill_between(train_sizes,
                 train_scores.mean(axis=1) - train_scores.std(axis=1),
                 train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1, color='blue')
plt.fill_between(train_sizes,
                 val_scores.mean(axis=1) - val_scores.std(axis=1),
                 val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.1, color='orange')
plt.title('Logistic Regression – Learning Curve')
plt.xlabel('Training Size')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
svm = SVC(random_state=42)
svm.fit(X_train, y_train)

svm_train_acc = accuracy_score(y_train, svm.predict(X_train))
svm_val_acc   = accuracy_score(y_val,   svm.predict(X_val))

print("=" * 50)
print("Model : SVM")
print(f"Train Accuracy : {svm_train_acc:.4f}")
print(f"Val   Accuracy : {svm_val_acc:.4f}")
print(classification_report(y_val, svm.predict(X_val)))

In [ ]:
cm = confusion_matrix(y_val, svm.predict(X_val))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('SVM – Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# ── Decision Tree ────────────────────────────────────────────────────────────
dt = DecisionTreeClassifier(random_state=42, max_depth=5)
dt.fit(X_train, y_train)

dt_train_acc = accuracy_score(y_train, dt.predict(X_train))
dt_val_acc   = accuracy_score(y_val,   dt.predict(X_val))

print("=" * 50)
print("Model : Decision Tree")
print(f"Train Accuracy : {dt_train_acc:.4f}")
print(f"Val   Accuracy : {dt_val_acc:.4f}")
print(classification_report(y_val, dt.predict(X_val)))



In [ ]:
cm = confusion_matrix(y_val, dt.predict(X_val))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Decision Tree – Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
rf = RandomForestClassifier(random_state=42,max_depth=5)
rf.fit(X_train, y_train)

rf_train_acc = accuracy_score(y_train, rf.predict(X_train))
rf_val_acc   = accuracy_score(y_val,   rf.predict(X_val))

print("=" * 50)
print("Model : Random Forest")
print(f"Train Accuracy : {rf_train_acc:.4f}")
print(f"Val   Accuracy : {rf_val_acc:.4f}")
print(classification_report(y_val, rf.predict(X_val)))

In [ ]:
cm = confusion_matrix(y_val, rf.predict(X_val))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Random Forest – Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Find best K
k_range = range(1, 21)
train_scores = []
val_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, knn.predict(X_train)))
    val_scores.append(accuracy_score(y_val, knn.predict(X_val)))

# Plot
plt.figure(figsize=(10, 5))
plt.plot(k_range, train_scores, label='Train Accuracy', color='blue')
plt.plot(k_range, val_scores,   label='Val Accuracy',   color='orange')
plt.title('KNN – Finding Best K')
plt.xlabel('K')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

best_k = val_scores.index(max(val_scores)) + 1
print(f"Best K: {best_k} → Val Accuracy: {max(val_scores):.4f}")

# Train with best K
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train, y_train)

knn_train_acc = accuracy_score(y_train, knn.predict(X_train))
knn_val_acc   = accuracy_score(y_val,   knn.predict(X_val))

print("=" * 50)
print("Model : KNN")
print(f"Train Accuracy : {knn_train_acc:.4f}")
print(f"Val   Accuracy : {knn_val_acc:.4f}")
print(classification_report(y_val, knn.predict(X_val)))

In [ ]:
cm = confusion_matrix(y_val, knn.predict(X_val))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('KNN – Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

k_range = range(1, 21, 2) 
cv_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    
    scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())


best_k = k_range[np.argmax(cv_scores)]
print(f"Best K from CV: {best_k}")

import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report

plt.figure(figsize=(10, 6))
plt.plot(k_range, cv_scores, marker='o', linestyle='dashed', color='green', markerfacecolor='red')
plt.title('Accuracy Score vs. K Value (Cross-Validation)')
plt.xlabel('K')
plt.ylabel('Mean CV Accuracy')
plt.xticks(k_range)
plt.grid(True)
plt.show()

print(f"--- Final Model Training with Best K: {best_k} ---")
final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train, y_train)

y_pred = final_knn.predict(X_val)

final_acc = accuracy_score(y_val, y_pred)
print("=" * 50)
print(f"Final Validation Accuracy: {final_acc:.4f}")
print("-" * 50)
print("Detailed Classification Report:")
print(classification_report(y_val, y_pred))

In [ ]:
# ── Logistic Regression Tuning ───────────────────────────────────────────────
lr_params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}
lr_grid = GridSearchCV(LogisticRegression(random_state=42, max_iter=1000),
                       lr_params, cv=4, scoring='accuracy', n_jobs=-1)
lr_grid.fit(X_train, y_train)
print("Best LR params:", lr_grid.best_params_)
print("Best CV score :", round(lr_grid.best_score_, 4))

lr_best = lr_grid.best_estimator_
lr_best_train_acc = accuracy_score(y_train, lr_best.predict(X_train))
lr_best_val_acc   = accuracy_score(y_val,   lr_best.predict(X_val))

print("=" * 50)
print("Model : Logistic Regression (Tuned)")
print(f"Train Accuracy : {lr_best_train_acc:.4f}")
print(f"Val   Accuracy : {lr_best_val_acc:.4f}")
print(classification_report(y_val, lr_best.predict(X_val)))

In [ ]:
cm = confusion_matrix(y_val, lr_best.predict(X_val))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Logistic Regression (Tuned) – Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# ── SVM Tuning ───────────────────────────────────────────────────────────────
svm_params = {
    'C': [0.1, 1, 10],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto']
}
svm_grid = GridSearchCV(SVC(random_state=42),
                        svm_params, cv=5, scoring='accuracy', n_jobs=-1)
svm_grid.fit(X_train, y_train)
print("Best SVM params:", svm_grid.best_params_)
print("Best CV score  :", round(svm_grid.best_score_, 4))

svm_best = svm_grid.best_estimator_
svm_best_train_acc = accuracy_score(y_train, svm_best.predict(X_train))
svm_best_val_acc   = accuracy_score(y_val,   svm_best.predict(X_val))

print("=" * 50)
print("Model : SVM (Tuned)")
print(f"Train Accuracy : {svm_best_train_acc:.4f}")
print(f"Val   Accuracy : {svm_best_val_acc:.4f}")
print(classification_report(y_val, svm_best.predict(X_val)))



In [ ]:
cm = confusion_matrix(y_val, svm_best.predict(X_val))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('SVM (Tuned) – Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# ── Decision Tree Tuning ──────────────────────────────────────────────────────
# Test different max_depth values to find the optimal balance
depth_range = range(2, 16)
#over 16 caouse overfitting, so i stopped at 16
dt_train_scores = []
dt_val_scores = []

for depth in depth_range:
    dt_temp = DecisionTreeClassifier(random_state=42, max_depth=depth)
    dt_temp.fit(X_train, y_train)
    dt_train_scores.append(accuracy_score(y_train, dt_temp.predict(X_train)))
    dt_val_scores.append(accuracy_score(y_val, dt_temp.predict(X_val)))

# Plot to find optimal depth

plt.figure(figsize=(10, 5))
plt.plot(depth_range, dt_train_scores, label='Train Accuracy', color='blue', marker='o')
plt.plot(depth_range, dt_val_scores,   label='Val Accuracy',   color='orange', marker='s')
plt.title('Decision Tree – Finding Optimal Max Depth')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Find the optimal depth
optimal_depth = depth_range[dt_val_scores.index(max(dt_val_scores))]
print(f"Optimal Max Depth: {optimal_depth}")

# Train the final model with the optimal depth
dt_best = DecisionTreeClassifier(random_state=42, max_depth=optimal_depth)
dt_best.fit(X_train, y_train)

dt_best_train_acc = accuracy_score(y_train, dt_best.predict(X_train))
dt_best_val_acc   = accuracy_score(y_val,   dt_best.predict(X_val))

print("=" * 50)
print("Model : Decision Tree (Tuned)")
print(f"Train Accuracy : {dt_best_train_acc:.4f}")
print(f"Val   Accuracy : {dt_best_val_acc:.4f}")
print(f"Gap (Overfitting): {dt_best_train_acc - dt_best_val_acc:.4f}")
print(classification_report(y_val, dt_best.predict(X_val)))

In [ ]:
cm = confusion_matrix(y_val, dt_best.predict(X_val))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Decision Tree (Tuned) – Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# ── Model Comparison ──────────────────────────────────────────────────────────
results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Logistic Regression (Tuned)',
        'SVM',
        'SVM (Tuned)',
        'Decision Tree',
        'Decision Tree (Tuned)',
        'Random Forest',
        'KNN'
    ],
    'Train Accuracy': [
        lr_train_acc,
        lr_best_train_acc,
        svm_train_acc,
        svm_best_train_acc,
        dt_train_acc,
        dt_best_train_acc,
        rf_train_acc,
        knn_train_acc
    ],
    'Val Accuracy': [
        lr_val_acc,
        lr_best_val_acc,
        svm_val_acc,
        svm_best_val_acc,
        dt_val_acc,
        dt_best_val_acc,
        rf_val_acc,
        knn_val_acc
    ]
}).sort_values('Val Accuracy', ascending=False).reset_index(drop=True)

print(results.to_string(index=False))

# Graph
fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(results))
width = 0.35

bars1 = ax.barh([i + width/2 for i in x], results['Train Accuracy'], width, label='Train Accuracy', color='steelblue')
bars2 = ax.barh([i - width/2 for i in x], results['Val Accuracy'],   width, label='Val Accuracy',   color='orange')

ax.axvline(x=0.80, color='red', linestyle='--', label='80% threshold')
ax.set_yticks(list(x))
ax.set_yticklabels(results['Model'])
ax.set_xlabel('Accuracy')
ax.set_title('Model Comparison – Train vs Val Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Auto-select Best Model ─────────────────────────────────────────────────────
models_dict = {
    'Logistic Regression':        lr,
    'Logistic Regression (Tuned)': lr_best,
    'SVM':                        svm,
    'SVM (Tuned)':                svm_best,
    'Decision Tree':              dt,
    'Decision Tree (Tuned)':      dt_best,
    'Random Forest':              rf,
    'KNN':                        knn
}

val_accs = {
    'Logistic Regression':        lr_val_acc,
    'Logistic Regression (Tuned)': lr_best_val_acc,
    'SVM':                        svm_val_acc,
    'SVM (Tuned)':                svm_best_val_acc,
    'Decision Tree':              dt_val_acc,
    'Decision Tree (Tuned)':      dt_best_val_acc,
    'Random Forest':              rf_val_acc,
    'KNN':                        knn_val_acc
}

best_name  = max(val_accs, key=val_accs.get)
best_model = models_dict[best_name]
print(f"\n Best Model: {best_name} ({round(val_accs[best_name], 4)})")

In [ ]:
print("X_train columns:", X_train.columns.tolist())
print("test columns:", test.columns.tolist())

In [ ]:
test_preds = best_model.predict(test)
print("Prediction distribution:", pd.Series(test_preds).value_counts().to_dict())

submission = pd.DataFrame({'Prediction': test_preds})
submission.to_csv('titanic_predictions.csv', index=False)
print("Saved titanic_predictions.csv ")
submission.head(10)

In [ ]:
result_df = test.copy()
result_df['Prediction'] = test_preds
result_df['Prediction'] = result_df['Prediction'].map({0: 'Died', 1: 'Survived'})

result_df.to_csv('titanic_predictions.csv', index=False)
print("Saved titanic_predictions.csv ")
result_df.head(20)

# "I hope this project meets your expectations.Shosha,Amr,Honda  "